# Truc quan hoa ket qua (ban cho Colab)

So tay nay uu tien chay on trong Colab va cung chay duoc tren may local.

Noi dung:
1. Xac dinh thu muc du an
2. Doc ket qua RFM
3. Ve bieu do RFM
4. Doc va ve top luat MBA

In [ ]:
from pathlib import Path
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1) Xac dinh thu muc du an

In [ ]:
def dang_colab() -> bool:
    return 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())

def tim_root_du_an() -> Path:
    cwd = Path.cwd().resolve()
    ung_vien = [cwd, cwd.parent]

    content = Path('/content')
    if content.exists():
        for p in content.iterdir():
            if p.is_dir():
                ung_vien.append(p.resolve())

    for root in ung_vien:
        if (root / 'src').exists() and (root / 'data').exists():
            return root

    return cwd

ROOT = tim_root_du_an()

if dang_colab() and not (ROOT / 'src').exists():
    print('Ban dang o Colab nhung chua co repo du an.')
    print('Hay clone repo roi chay lai cell nay.')
    print('Vi du: !git clone https://github.com/phoudsavanhKongmany/final-bigdata-project-nhom9Test.git')

RFM_SEGMENT_PATH = ROOT / 'data/3_curated/results/rfm/customer_segments'
RFM_SUMMARY_PATH = ROOT / 'data/3_curated/results/rfm/segment_summary'
MBA_RULES_PATH = ROOT / 'data/3_curated/results/mba/association_rules'

print('ROOT:', ROOT)
print('Co tep phan khuc RFM:', RFM_SEGMENT_PATH.exists())
print('Co tep tong hop RFM:', RFM_SUMMARY_PATH.exists())
print('Co tep luat MBA:', MBA_RULES_PATH.exists())

## 2) Doc ket qua RFM

In [ ]:
rfm_segments = None
rfm_summary = None

if RFM_SEGMENT_PATH.exists() and RFM_SUMMARY_PATH.exists():
    rfm_segments = pd.read_parquet(RFM_SEGMENT_PATH)
    rfm_summary = pd.read_parquet(RFM_SUMMARY_PATH)

    print('Da doc du lieu RFM thanh cong.')
    display(rfm_segments.head())

    if 'business_score' in rfm_summary.columns:
        display(rfm_summary.sort_values('business_score', ascending=False))
    elif 'avg_monetary' in rfm_summary.columns:
        display(rfm_summary.sort_values('avg_monetary', ascending=False))
    else:
        display(rfm_summary)
else:
    print('Chua co ket qua RFM.')
    print('Hay chay: python3 src/main_pipeline.py --project rfm --step all')

## 3) Ve bieu do RFM

In [ ]:
if rfm_segments is not None and not rfm_segments.empty and 'segment_label' in rfm_segments.columns:
    bang_dem = (
        rfm_segments['segment_label']
        .value_counts()
        .rename_axis('segment_label')
        .reset_index(name='customers')
    )

    ax = sns.barplot(
        data=bang_dem,
        x='segment_label',
        y='customers',
        hue='segment_label',
        palette='Set2',
        legend=False
    )
    ax.set_title('So luong khach hang theo phan khuc')
    ax.set_xlabel('Phan khuc')
    ax.set_ylabel('So khach hang')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print('Khong co du lieu phan khuc de ve bieu do.')

In [ ]:
if rfm_summary is not None and not rfm_summary.empty:
    ds_chi_so = [c for c in ['avg_recency_days', 'avg_frequency', 'avg_monetary'] if c in rfm_summary.columns]

    if ds_chi_so and 'segment_label' in rfm_summary.columns:
        metrics_long = rfm_summary.melt(
            id_vars=['segment_label'],
            value_vars=ds_chi_so,
            var_name='chi_so',
            value_name='gia_tri'
        )

        g = sns.catplot(
            data=metrics_long,
            x='segment_label',
            y='gia_tri',
            col='chi_so',
            kind='bar',
            sharey=False,
            palette='Set3',
            height=4,
            aspect=1.1
        )
        g.set_titles('{col_name}')
        for truc in g.axes.flat:
            truc.tick_params(axis='x', rotation=20)
        plt.tight_layout()
        plt.show()
    else:
        print('Khong du cot de ve bieu do chi so RFM.')
else:
    print('Khong co bang tong hop RFM de ve bieu do.')

## 4) Doc va ve cac luat MBA

In [ ]:
mba_rules = None

if MBA_RULES_PATH.exists():
    mba_rules = pd.read_parquet(MBA_RULES_PATH)

    if mba_rules.empty:
        print('Tep luat ket hop ton tai nhung rong.')
    else:
        top_rules = mba_rules.sort_values('confidence', ascending=False).head(15).copy()

        if 'antecedent' in top_rules.columns and 'consequent' in top_rules.columns:
            top_rules['rule'] = top_rules['antecedent'].astype(str) + ' => ' + top_rules['consequent'].astype(str)
        elif 'antecedents' in top_rules.columns and 'consequents' in top_rules.columns:
            top_rules['rule'] = top_rules['antecedents'].astype(str) + ' => ' + top_rules['consequents'].astype(str)
        else:
            top_rules['rule'] = top_rules.index.astype(str)

        cols_hien = [c for c in ['rule', 'confidence', 'lift', 'support'] if c in top_rules.columns]
        display(top_rules[cols_hien])

        plt.figure(figsize=(12, 7))
        sns.barplot(
            data=top_rules,
            x='confidence',
            y='rule',
            hue='rule',
            palette='viridis',
            legend=False
        )
        plt.title('Top 15 luat ket hop theo do tin cay')
        plt.xlabel('Confidence')
        plt.ylabel('Luat')
        plt.tight_layout()
        plt.show()
else:
    print('Chua co ket qua MBA.')
    print('Hay chay: python3 src/main_pipeline.py --project mba --step all')